# Two-Stage Evidence Retrieval and Transformer-Based Claim Verification


## Overview

This project was submitted by Team 32 for Natural Language Processing at the
University of Melbourne. Candidate retrieval and evidence reranking form a
two-stage retrieval system, followed by a DeBERTa claim classifier.

1. Build word and character TF-IDF indexes and retrieve 500 passages per claim.
2. Rerank the candidates with a pretrained MiniLM cross-encoder and choose the
   evidence depth using development-set retrieval F1.
3. Fine-tune DeBERTa-v3 with class-weighted cross-entropy on labelled training
   claims and gold evidence. Evaluate the pipeline using retrieved evidence.

Read `docs/evaluation.md` before interpreting results. The 89% figure refers to
at least one gold passage appearing among 500 candidates, not claim accuracy.
Development results are used for model selection and are not held-out test results.

The optional train+development fit is disabled by default. If enabled, it uses
the previously selected epoch count without evaluating on data included in that
fit. Historical output logs remain in a separate local archive.


# 1. Data Preparation


## Install Python packages


In [ ]:
# Set True only when dependencies have not been installed in this environment.
INSTALL_DEPENDENCIES = False

import os
import subprocess
import sys
from pathlib import Path

setup_root = Path(os.environ.get(
    'CLAIM_PROJECT_ROOT', '..' if Path.cwd().name == 'notebooks' else '.'
)).resolve()
requirements = setup_root / 'requirements.txt'
if not requirements.is_file():
    raise FileNotFoundError(
        'Open this notebook from the repository or set CLAIM_PROJECT_ROOT.'
    )
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(requirements)])
else:
    print('Using the current environment. See README.md for dependency installation.')


In [ ]:
import os, sys, json, re, time, math, random, pickle, string, gc, unicodedata
from pathlib import Path
from typing import Any, Dict, List, Tuple, Sequence, Optional, Iterable

import joblib
import numpy as np
import torch
from torch.utils.data import DataLoader
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm import tqdm

if 'NOTEBOOK_START_TIME' not in globals():
    NOTEBOOK_START_TIME = time.time()

def elapsed_hours(start_time=None) -> float:
    start_time = NOTEBOOK_START_TIME if start_time is None else start_time
    return (time.time() - start_time) / 3600.0

def print_elapsed_hours(tag='[TIME] elapsed') -> None:
    hours = elapsed_hours()
    print(f'{tag}: {hours:.4f} hours ({hours * 60:.1f} minutes)')

print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
print('imports ready')


## Configure Paths and Hyperparameters

Set the data paths, model names, retrieval settings, and output files.


In [ ]:
# ---------- Paths -----------------------------------------------------------
PROJECT_ROOT = Path(os.environ.get(
    'CLAIM_PROJECT_ROOT', '..' if Path.cwd().name == 'notebooks' else '.'
)).resolve()
DATA_DIR     = PROJECT_ROOT / 'data'
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

EVIDENCE_FILE = DATA_DIR / 'evidence.json'
TRAIN_FILE    = DATA_DIR / 'train-claims.json'
DEV_FILE      = DATA_DIR / 'dev-claims.json'
TEST_FILE     = DATA_DIR / 'test-claims-unlabelled.json'

# ---------- Stage-1 retrieval: team TF-IDF hybrid ---------------------------
STAGE1_METHOD = 'tfidf_hybrid_word_char'
STAGE1_CANDIDATE_POOL_SIZE = 500
CANDIDATE_POOL_SIZE_GRID = [1, 3, 5, 8, 10, 20, 50, 100, 200, 300, 500]

WORD_NGRAM_RANGE = (1, 2)
WORD_MIN_DF = 2
WORD_MAX_DF = 0.95
WORD_MAX_FEATURES = 600_000

CHAR_ANALYZER = 'char_wb'
CHAR_NGRAM_RANGE = (3, 5)
CHAR_MIN_DF = 3
CHAR_MAX_DF = 0.95
CHAR_MAX_FEATURES = 800_000

WORD_WEIGHT = 1.0
CHAR_WEIGHT = 0.7

STAGE1_DIR = ARTIFACT_DIR / 'stage1_tfidf'
STAGE1_DIR.mkdir(parents=True, exist_ok=True)
WORD_VECTORIZER_FILE = STAGE1_DIR / 'word_tfidf_vectorizer.joblib'
WORD_MATRIX_FILE = STAGE1_DIR / 'word_tfidf_matrix.npz'
CHAR_VECTORIZER_FILE = STAGE1_DIR / 'char_tfidf_vectorizer.joblib'
CHAR_MATRIX_FILE = STAGE1_DIR / 'char_tfidf_matrix.npz'
STAGE1_EVIDENCE_IDS_FILE = STAGE1_DIR / 'evidence_ids.json'
STAGE1_META_FILE = STAGE1_DIR / 'stage1_tfidf_meta.json'

DEV_STAGE1_SCORED_FILE = STAGE1_DIR / 'dev_stage1_tfidf500_scored.json'
TRAIN_STAGE1_SCORED_FILE = STAGE1_DIR / 'train_stage1_tfidf500_scored.json'
TEST_STAGE1_SCORED_FILE = STAGE1_DIR / 'test_stage1_tfidf500_scored.json'
DEV_STAGE1_METRICS_FILE = STAGE1_DIR / 'dev_stage1_tfidf500_metrics.json'

FORCE_REBUILD_STAGE1    = False
FORCE_REBUILD_RETRIEVAL = False  # reuse cached retrieval when available; rebuild only if files are missing

# ---------- Stage-2 retrieval: cross-encoder reranker -----------------------
K_RETRIEVE_CHOICES = [2, 3, 4, 5, 6, 7]
K_RETRIEVE    = 3      # fallback; overwritten by dev retrieval F1 sweep below
MAX_K_RETRIEVE = max(K_RETRIEVE_CHOICES)
CE_BASE_MODEL = 'cross-encoder/ms-marco-MiniLM-L6-v2'  # pretrained L6 reranker; no fine-tuning
CE_MODEL      = CE_BASE_MODEL
CE_BATCH_SIZE = 256
CE_MAX_LEN    = 192
CE_EVIDENCE_CHAR_LIMIT = 1200  # truncate very long evidence before CE tokenization for speed
CE_RERANK_WEIGHT = 1.0   # pure pretrained L6 reranking over the TF-IDF 500 candidates

RETRIEVED_DEV_FILES_BY_K   = {k: ARTIFACT_DIR / f'retrieved_dev_tfidf500_l6_top{k}.json' for k in K_RETRIEVE_CHOICES}
RETRIEVED_TEST_FILES_BY_K  = {k: ARTIFACT_DIR / f'retrieved_test_tfidf500_l6_top{k}.json' for k in K_RETRIEVE_CHOICES}
RETRIEVED_DEV_FILE   = RETRIEVED_DEV_FILES_BY_K[K_RETRIEVE]
RETRIEVED_TEST_FILE  = RETRIEVED_TEST_FILES_BY_K[K_RETRIEVE]

# Gold classifier examples keep all labelled gold evidence.
GOLD_EVIDENCE_MAX_PIECES = 10

# ---------- Classifier hyper-parameters -------------------------------------
# Reduce the model or sequence length if GPU memory is insufficient.
CLS_MODEL  = 'microsoft/deberta-v3-base'
LABELS     = ['SUPPORTS', 'REFUTES', 'NOT_ENOUGH_INFO', 'DISPUTED']
LABEL2ID   = {l: i for i, l in enumerate(LABELS)}
ID2LABEL   = {i: l for i, l in enumerate(LABELS)}

CLASSIFIER_DIR        = ARTIFACT_DIR / 'classifier'
FINAL_CLASSIFIER_DIR  = ARTIFACT_DIR / 'classifier_final'
DEV_PREDICTIONS_FILE  = ARTIFACT_DIR / 'dev-claims-predictions.json'
TEST_PREDICTIONS_FILE = ARTIFACT_DIR / 'test-output.json'
SUBMISSION_ZIP_FILE   = ARTIFACT_DIR / 'submission.zip'

MAX_LEN          = 512
EPOCHS           = 15
LR               = 3e-5
BATCH_SIZE       = 1
EVAL_BATCH_SIZE  = 8
WEIGHT_DECAY     = 0.01
WARMUP_RATIO     = 0.08
GRAD_ACCUM_STEPS = 16

# ---------- Misc ------------------------------------------------------------
SEED = 20260517
RUN_TEST_SUBMISSION = False  # Enable only for the optional final train+dev fit and test predictions.
DOWNLOAD_DATA = False  # Read data/README.md, then opt in to the original public-source downloader.

def set_all_seeds(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

set_all_seeds()
print('config loaded; LABELS =', LABELS)
print('artifacts ->', ARTIFACT_DIR)
print('stage 1 method ->', STAGE1_METHOD)
print('stage 1 candidate pool ->', STAGE1_CANDIDATE_POOL_SIZE)


## Load Data and Define Utilities

Place the documented source files in `data/` or explicitly enable `DOWNLOAD_DATA`.


In [ ]:
# Robust JSON/text utilities. This cell can run even if earlier imports were reset.
import json
import re
import string
from pathlib import Path
from typing import Dict, Any

try:
    import ujson as _ujson
except Exception:
    _ujson = None


def load_json(path) -> dict:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Missing required file: {path}')
    with path.open('r', encoding='utf-8') as fh:
        return _ujson.load(fh) if _ujson is not None else json.load(fh)


def save_json(obj, path, indent=2) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as fh:
        json.dump(obj, fh, ensure_ascii=False, indent=indent)
        fh.write('\n')


def load_evidence(path=None) -> Dict[str, str]:
    path = EVIDENCE_FILE if path is None else path
    evidence = load_json(path)
    if not evidence:
        raise ValueError(f'Evidence file is empty: {path}')
    return {str(eid): str(text) for eid, text in evidence.items()}


def load_claims(path) -> Dict[str, dict]:
    claims = load_json(path)
    if not claims:
        raise ValueError(f'Claim file is empty: {path}')
    return {str(cid): claim for cid, claim in claims.items()}


_WS_RE = re.compile(r'\s+')
_PUNCT_TABLE = str.maketrans({c: ' ' for c in string.punctuation})


def clean_text(text: Any) -> str:
    if text is None:
        return ''
    return _WS_RE.sub(' ', str(text)).strip()


def tokenize(text: Any) -> list:
    text = clean_text(text)
    if not text:
        return []
    text = text.lower().translate(_PUNCT_TABLE)
    return [token for token in text.split() if token]


def validate_submission(claims: Dict[str, dict], submission: Dict[str, dict]) -> None:
    missing = sorted(set(claims) - set(submission))
    extra = sorted(set(submission) - set(claims))
    if missing:
        raise ValueError(f'missing {len(missing)} predictions; first missing id: {missing[0]}')
    if extra:
        raise ValueError(f'found {len(extra)} unknown predictions; first unknown id: {extra[0]}')
    for cid, item in submission.items():
        label = item.get('claim_label')
        if label not in LABEL2ID:
            raise ValueError(f'{cid}: invalid claim_label={label!r}')
        evidences = item.get('evidences')
        if not isinstance(evidences, list) or not evidences:
            raise ValueError(f'{cid}: evidences must be a non-empty list')
        if not all(isinstance(evidence_id, str) for evidence_id in evidences):
            raise ValueError(f'{cid}: every evidence id must be a string')
    print(f'[CHECK] {len(submission)} predictions validated.')


print('utilities ready')


# Download the official assignment files if they are not already present.
# This lets the notebook run from a fresh Colab session without manually uploading data.
import subprocess
import urllib.request

DATA_DIR.mkdir(parents=True, exist_ok=True)

RAW_BASE = 'https://raw.githubusercontent.com/drcarenhan/COMP90042_2026/main'
REQUIRED_SMALL_FILES = {
    TRAIN_FILE: f'{RAW_BASE}/data/train-claims.json',
    DEV_FILE: f'{RAW_BASE}/data/dev-claims.json',
    TEST_FILE: f'{RAW_BASE}/data/test-claims-unlabelled.json',
    DATA_DIR / 'dev-claims-baseline.json': f'{RAW_BASE}/data/dev-claims-baseline.json',
}

for local_path, url in REQUIRED_SMALL_FILES.items():
    local_path = Path(local_path)
    if local_path.exists() and local_path.stat().st_size > 0:
        print(f'[DATA] exists: {local_path}')
        continue
    if not DOWNLOAD_DATA:
        raise FileNotFoundError(f'Place {local_path.name} in data/ or enable DOWNLOAD_DATA after reviewing data/README.md.')
    local_path.parent.mkdir(parents=True, exist_ok=True)
    print(f'[DATA] downloading {url} -> {local_path}')
    urllib.request.urlretrieve(url, local_path)

if EVIDENCE_FILE.exists() and EVIDENCE_FILE.stat().st_size > 0:
    print(f'[DATA] exists: {EVIDENCE_FILE}')
else:
    if not DOWNLOAD_DATA:
        raise FileNotFoundError('Place evidence.json in data/ or explicitly enable DOWNLOAD_DATA.')
    print('[DATA] downloading evidence.json from course Google Drive link ...')
    try:
        subprocess.run([
            sys.executable,
            '-m',
            'gdown',
            '1JlUzRufknsHzKzvrEjgw8D3n_IRpjzo6',
            '-O',
            str(EVIDENCE_FILE),
        ], check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            'Could not download evidence.json with gdown. Upload or place '
            'data/evidence.json manually from the official assignment Google Drive link.'
        ) from exc

# Sanity check: load files and show split sizes.
for path in [TRAIN_FILE, DEV_FILE, TEST_FILE, EVIDENCE_FILE]:
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        raise FileNotFoundError(f'Missing or empty required file: {path}')

_train = load_claims(TRAIN_FILE)
_dev = load_claims(DEV_FILE)
_test = load_claims(TEST_FILE)
_evidence = load_evidence(EVIDENCE_FILE)
print(f'[DATA] train={len(_train):,}, dev={len(_dev):,}, test={len(_test):,}, evidence={len(_evidence):,}')
print('[DATA] ready')


## Stage 1: Candidate Evidence Retrieval (TF-IDF)

Build a hybrid word/character TF-IDF index and retrieve 500 candidate evidence passages for each claim.


In [ ]:
_WS_RE = re.compile(r'\s+')
_SUBSCRIPT_DIGITS = str.maketrans({chr(0x2080 + i): str(i) for i in range(10)})
_DEGREE_SIGN = chr(176)


def normalize_for_stage1(text: Any) -> str:
    text = clean_text(text)
    text = unicodedata.normalize('NFKC', text).translate(_SUBSCRIPT_DIGITS).lower()

    # Keep this cell ASCII-safe: construct Unicode punctuation with code points.
    text = text.replace(chr(0x2013), '-').replace(chr(0x2014), '-').replace(chr(0x2212), '-')
    text = text.replace(chr(0x2019), "'").replace('`', "'")

    text = re.sub(r'\bco\s*[- ]?\s*2\b', ' co2 ', text)
    text = re.sub(r'\bcarbon\s+dioxide\b', ' carbon dioxide co2 ', text)
    text = re.sub(r'\bch\s*[- ]?\s*4\b', ' ch4 methane ', text)
    text = re.sub(r'\bn\s*[- ]?\s*2\s*o\b', ' n2o nitrous oxide ', text)

    celsius_pattern = rf'(\d+(?:\.\d+)?)\s*(?:{_DEGREE_SIGN}\s*c|degrees?\s+c(?:elsius)?|deg\s*c|celsius)\b'
    fahrenheit_pattern = rf'(\d+(?:\.\d+)?)\s*(?:{_DEGREE_SIGN}\s*f|degrees?\s+f(?:ahrenheit)?|deg\s*f|fahrenheit)\b'
    text = re.sub(celsius_pattern, r' \1c \1 celsius ', text)
    text = re.sub(fahrenheit_pattern, r' \1f \1 fahrenheit ', text)
    text = re.sub(r'(\d+(?:\.\d+)?)\s*ppm\b', r' \1ppm \1 ppm ', text)
    text = re.sub(r'(\d+(?:\.\d+)?)\s*(?:%|per\s*cent|percent)\b', r' \1percent \1 percent ', text)

    replacements = {
        r'\bpre[- ]industrial\b': ' preindustrial pre industrial ',
        r'\bsea[- ]level\b': ' sea_level sea level ',
        r'\bglobal[- ]warming\b': ' global_warming global warming ',
        r'\bclimate[- ]sensitivity\b': ' climate_sensitivity climate sensitivity ',
        r'\bice[- ]sheet\b': ' ice_sheet ice sheet ',
        r'\bfossil[- ]fuel\b': ' fossil_fuel fossil fuel ',
        r'\bgreenhouse[- ]gas(?:es)?\b': ' greenhouse_gas greenhouse gas ',
    }
    for pattern, repl in replacements.items():
        text = re.sub(pattern, repl, text)
    return clean_text(text)


def stage1_meta() -> dict:
    return {
        'stage1_method': STAGE1_METHOD,
        'word_ngram_range': list(WORD_NGRAM_RANGE),
        'word_min_df': WORD_MIN_DF,
        'word_max_df': WORD_MAX_DF,
        'word_max_features': WORD_MAX_FEATURES,
        'char_analyzer': CHAR_ANALYZER,
        'char_ngram_range': list(CHAR_NGRAM_RANGE),
        'char_min_df': CHAR_MIN_DF,
        'char_max_df': CHAR_MAX_DF,
        'char_max_features': CHAR_MAX_FEATURES,
        'word_weight': WORD_WEIGHT,
        'char_weight': CHAR_WEIGHT,
        'evidence_file': str(EVIDENCE_FILE),
    }


def tfidf_cache_valid() -> bool:
    required = [
        WORD_VECTORIZER_FILE,
        WORD_MATRIX_FILE,
        CHAR_VECTORIZER_FILE,
        CHAR_MATRIX_FILE,
        STAGE1_EVIDENCE_IDS_FILE,
        STAGE1_META_FILE,
    ]
    if not all(path.exists() for path in required):
        return False
    cached = load_json(STAGE1_META_FILE)
    expected = stage1_meta()
    # The vectorizer/matrix cache does not depend on candidate_pool_size.
    # Older notebooks stored that value in metadata, so ignore it for cache compatibility.
    cached.pop('stage1_candidate_pool_size', None)
    expected.pop('stage1_candidate_pool_size', None)
    return cached == expected


def load_evidence_lists() -> Tuple[List[str], List[str]]:
    evidence = load_evidence(EVIDENCE_FILE)
    evidence_ids = list(evidence.keys())
    evidence_texts = [normalize_for_stage1(evidence[eid]) for eid in evidence_ids]
    return evidence_ids, evidence_texts


def build_or_load_tfidf(force_rebuild: bool = False):
    if tfidf_cache_valid() and not force_rebuild:
        print('[STAGE1] reusing cached TF-IDF indexes')
        evidence_ids = load_json(STAGE1_EVIDENCE_IDS_FILE)
        word_vectorizer = joblib.load(WORD_VECTORIZER_FILE)
        word_matrix = sparse.load_npz(WORD_MATRIX_FILE)
        char_vectorizer = joblib.load(CHAR_VECTORIZER_FILE)
        char_matrix = sparse.load_npz(CHAR_MATRIX_FILE)
        return evidence_ids, word_vectorizer, word_matrix, char_vectorizer, char_matrix

    print('[STAGE1] loading and normalizing evidence')
    t0 = time.time()
    evidence_ids, evidence_texts = load_evidence_lists()
    print(f'[STAGE1] evidence passages: {len(evidence_texts):,}; loaded in {time.time() - t0:.1f}s')

    print('[STAGE1] fitting word TF-IDF')
    t0 = time.time()
    word_vectorizer = TfidfVectorizer(
        lowercase=False,
        token_pattern=r'(?u)\b[a-zA-Z0-9_][a-zA-Z0-9_.-]*\b',
        ngram_range=WORD_NGRAM_RANGE,
        min_df=WORD_MIN_DF,
        max_df=WORD_MAX_DF,
        max_features=WORD_MAX_FEATURES,
        sublinear_tf=True,
        norm='l2',
        dtype=np.float32,
    )
    word_matrix = word_vectorizer.fit_transform(evidence_texts)
    print(f'[STAGE1] word matrix {word_matrix.shape}, nnz={word_matrix.nnz:,}, built in {time.time() - t0:.1f}s')

    print('[STAGE1] fitting character TF-IDF')
    t0 = time.time()
    char_vectorizer = TfidfVectorizer(
        lowercase=False,
        analyzer=CHAR_ANALYZER,
        ngram_range=CHAR_NGRAM_RANGE,
        min_df=CHAR_MIN_DF,
        max_df=CHAR_MAX_DF,
        max_features=CHAR_MAX_FEATURES,
        sublinear_tf=True,
        norm='l2',
        dtype=np.float32,
    )
    char_matrix = char_vectorizer.fit_transform(evidence_texts)
    print(f'[STAGE1] char matrix {char_matrix.shape}, nnz={char_matrix.nnz:,}, built in {time.time() - t0:.1f}s')

    save_json(evidence_ids, STAGE1_EVIDENCE_IDS_FILE)
    save_json(stage1_meta(), STAGE1_META_FILE)
    joblib.dump(word_vectorizer, WORD_VECTORIZER_FILE)
    sparse.save_npz(WORD_MATRIX_FILE, word_matrix)
    joblib.dump(char_vectorizer, CHAR_VECTORIZER_FILE)
    sparse.save_npz(CHAR_MATRIX_FILE, char_matrix)
    print('[STAGE1] cached TF-IDF indexes')
    return evidence_ids, word_vectorizer, word_matrix, char_vectorizer, char_matrix


def topk_from_sparse_scores(scores, k: int) -> List[Tuple[int, float]]:
    scores = scores.tocsr()
    if scores.nnz == 0 or k <= 0:
        return []
    data = scores.data
    indices = scores.indices
    k = min(k, len(data))
    if k == len(data):
        order = np.argsort(-data)
    else:
        order = np.argpartition(-data, k - 1)[:k]
        order = order[np.argsort(-data[order])]
    return [(int(indices[i]), float(data[i])) for i in order]


def tfidf_search_one(query: str, vectorizer, matrix, k: int) -> List[Tuple[int, float]]:
    query_norm = normalize_for_stage1(query)
    q = vectorizer.transform([query_norm])
    scores = q @ matrix.T
    return topk_from_sparse_scores(scores, k)


def minmax_normalize(items: List[Tuple[int, float]]) -> Dict[int, float]:
    if not items:
        return {}
    values = np.array([score for _, score in items], dtype=np.float32)
    lo = float(values.min())
    hi = float(values.max())
    if hi <= lo:
        return {idx: 1.0 for idx, _ in items}
    return {idx: float((score - lo) / (hi - lo)) for idx, score in items}


def retrieve_stage1_candidates_for_claim(
    claim_text: str,
    evidence_ids: Sequence[str],
    word_vectorizer,
    word_matrix,
    char_vectorizer,
    char_matrix,
    pool_size: int = STAGE1_CANDIDATE_POOL_SIZE,
) -> List[dict]:
    word_items = tfidf_search_one(claim_text, word_vectorizer, word_matrix, pool_size)
    char_items = tfidf_search_one(claim_text, char_vectorizer, char_matrix, pool_size)
    word_norm = minmax_normalize(word_items)
    char_norm = minmax_normalize(char_items)
    merged = {}
    for idx, raw_score in word_items:
        merged.setdefault(idx, {'word_score': 0.0, 'char_score': 0.0, 'sources': []})
        merged[idx]['word_score'] = float(raw_score)
        merged[idx]['word_score_norm'] = word_norm.get(idx, 0.0)
        merged[idx]['sources'].append('word_tfidf')
    for idx, raw_score in char_items:
        merged.setdefault(idx, {'word_score': 0.0, 'char_score': 0.0, 'sources': []})
        merged[idx]['char_score'] = float(raw_score)
        merged[idx]['char_score_norm'] = char_norm.get(idx, 0.0)
        merged[idx]['sources'].append('char_tfidf')
    ranked = []
    for idx, item in merged.items():
        combined_score = WORD_WEIGHT * item.get('word_score_norm', 0.0) + CHAR_WEIGHT * item.get('char_score_norm', 0.0)
        ranked.append((idx, combined_score, item))
    ranked.sort(key=lambda x: x[1], reverse=True)
    out = []
    for rank, (idx, combined_score, item) in enumerate(ranked[:pool_size], start=1):
        out.append({
            'evidence_id': str(evidence_ids[idx]),
            'stage1_rank': rank,
            'stage1_score': float(combined_score),
            'stage1_method': STAGE1_METHOD,
            'word_score': float(item.get('word_score', 0.0)),
            'char_score': float(item.get('char_score', 0.0)),
            'sources': sorted(set(item.get('sources', []))),
        })
    return out


def candidates_to_id_only(scored_candidates: Dict[str, List[dict]]) -> Dict[str, List[str]]:
    return {claim_id: [item['evidence_id'] for item in candidates] for claim_id, candidates in scored_candidates.items()}


print('[STAGE1] team TF-IDF hybrid helpers ready')


In [ ]:
EVIDENCE_IDS, WORD_VECTORIZER, WORD_MATRIX, CHAR_VECTORIZER, CHAR_MATRIX = build_or_load_tfidf(
    force_rebuild=FORCE_REBUILD_STAGE1,
)
print(f'[STAGE1] index ready: evidence_ids={len(EVIDENCE_IDS):,}')


# 2. Model Implementation


## Stage 2: Evidence Reranking

Use a pretrained MiniLM cross-encoder to rerank the Stage 1 evidence candidates.


In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer


class CrossEncoderReranker:
    def __init__(self, model_name=CE_MODEL, max_length=CE_MAX_LEN,
                 batch_size=CE_BATCH_SIZE, fp16=True):
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.max_length = max_length
        self.batch_size = batch_size
        self.fp16 = fp16 and self.device == 'cuda'
        print(f'[CE] loading {model_name} on {self.device} ...')
        print(f'[CE] batch_size={self.batch_size} max_length={self.max_length}')
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name).to(self.device).eval()
        if self.fp16:
            self.model.half()
        torch.backends.cuda.matmul.allow_tf32 = True

    @torch.inference_mode()
    def score(self, pairs, show_progress=False, desc='stage2 rerank'):
        if not pairs:
            return np.zeros(0, dtype=np.float32)
        out = []
        starts = range(0, len(pairs), self.batch_size)
        if show_progress:
            starts = tqdm(starts, total=math.ceil(len(pairs) / self.batch_size), desc=desc)
        for start in starts:
            batch_pairs = pairs[start:start + self.batch_size]
            a = [p[0] for p in batch_pairs]
            b = [p[1] for p in batch_pairs]
            batch = self.tokenizer(
                a, b,
                padding=True,
                truncation=True,
                max_length=self.max_length,
                return_tensors='pt',
            )
            batch = {k: v.to(self.device, non_blocking=True) for k, v in batch.items()}
            logits = self.model(**batch).logits
            scores = logits.squeeze(-1) if logits.shape[-1] == 1 else logits.softmax(-1)[:, 1]
            out.append(scores.float().cpu().numpy())
        return np.concatenate(out, axis=0)


def ce_evidence_text(text: str) -> str:
    text = clean_text(text)
    limit = globals().get('CE_EVIDENCE_CHAR_LIMIT', None)
    if limit and len(text) > limit:
        return text[:limit]
    return text


print('Fast batched CrossEncoderReranker defined')


## Retrieve Dev Evidence for K Selection

Retrieve dev evidence once, save top-K variants, and reuse them for later evaluation.


In [ ]:
def _minmax(values):
    values = np.asarray(values, dtype=np.float32)
    if values.size == 0:
        return values
    lo = float(values.min())
    hi = float(values.max())
    if hi <= lo:
        return np.ones_like(values, dtype=np.float32)
    return (values - lo) / (hi - lo)


def run_retrieval(claims_path: Path,
                  out_path: Path,
                  evidence: Dict[str, str],
                  evidence_ids: Sequence[str],
                  word_vectorizer,
                  word_matrix,
                  char_vectorizer,
                  char_matrix,
                  reranker: Optional[CrossEncoderReranker] = None,
                  stage1_pool_size: int = STAGE1_CANDIDATE_POOL_SIZE,
                  k: int = K_RETRIEVE,
                  force: bool = False,
                  stage1_scored_out_path: Optional[Path] = None) -> Dict[str, list]:
    if k <= 0 or stage1_pool_size <= 0:
        raise ValueError('Evidence depth and candidate pool size must be positive.')
    if out_path.exists() and not force:
        print(f'[RET] reusing cached retrieval -> {out_path}')
        return load_json(out_path)

    print(f'[RET] {claims_path.name} using {STAGE1_METHOD} pool={stage1_pool_size}')
    claims = load_claims(claims_path)
    out: Dict[str, list] = {}
    stage1_scored: Dict[str, List[dict]] = {}
    pending = {}
    all_pairs = []
    t0 = time.time()

    # First collect all Stage-1 candidates. Then score all claim-evidence pairs in large CE batches.
    for cid, claim in tqdm(claims.items(), total=len(claims), desc='stage1 candidates'):
        candidates = retrieve_stage1_candidates_for_claim(
            claim_text=claim['claim_text'],
            evidence_ids=evidence_ids,
            word_vectorizer=word_vectorizer,
            word_matrix=word_matrix,
            char_vectorizer=char_vectorizer,
            char_matrix=char_matrix,
            pool_size=stage1_pool_size,
        )
        stage1_scored[cid] = candidates
        cand_ids = [item['evidence_id'] for item in candidates]
        if not cand_ids:
            raise ValueError(f'{cid}: no evidence candidates were retrieved. Check the claim and index.')

        if reranker is not None and CE_RERANK_WEIGHT > 0:
            start = len(all_pairs)
            all_pairs.extend(
                (claim['claim_text'], ce_evidence_text(evidence.get(eid, '')))
                for eid in cand_ids
            )
            pending[cid] = (
                start,
                len(cand_ids),
                cand_ids,
                np.asarray([item.get('stage1_score', 0.0) for item in candidates], dtype=np.float32),
            )
        else:
            out[cid] = cand_ids[:k]

    if reranker is not None and CE_RERANK_WEIGHT > 0 and all_pairs:
        print(f'[RET] Stage-2 scoring {len(all_pairs):,} pairs in batches of {reranker.batch_size}')
        all_scores = reranker.score(all_pairs, show_progress=True, desc=f'{claims_path.stem} L6 rerank')
        for cid, (start, n_items, cand_ids, stage1_scores) in pending.items():
            ce_scores = all_scores[start:start + n_items]
            ce_norm = _minmax(ce_scores)
            stage1_norm = _minmax(stage1_scores)
            blended = (1.0 - CE_RERANK_WEIGHT) * stage1_norm + CE_RERANK_WEIGHT * ce_norm
            order = blended.argsort()[::-1][:k]
            out[cid] = [cand_ids[i] for i in order]
        del all_scores, all_pairs
        gc.collect()

    print(f'[RET] done in {time.time() - t0:.1f}s')
    if stage1_scored_out_path is not None:
        save_json(stage1_scored, stage1_scored_out_path)
        print(f'[RET] wrote Stage-1 scored candidates -> {stage1_scored_out_path}')
    save_json(out, out_path)
    print(f'[RET] wrote final retrieved ids -> {out_path}')
    return out


print('[RET] fast batched run_retrieval helper ready')


In [ ]:
# Load evidence once and reuse for all retrieval calls.
print('Loading evidence corpus ...')
EVIDENCE_TEXT = load_evidence()
print(f'  evidence size = {len(EVIDENCE_TEXT):,}')

ACTIVE_CE_MODEL = CE_MODEL
DEV_RETRIEVAL_MAXK_FILE = RETRIEVED_DEV_FILES_BY_K[MAX_K_RETRIEVE]

if CE_RERANK_WEIGHT <= 0:
    print('[RERANKER] Stage-2 disabled; using Stage-1 TF-IDF ranking only.')
    reranker = None
else:
    print(f'[RERANKER] using pretrained reranker: {ACTIVE_CE_MODEL}')
    need_dev_reranker = FORCE_REBUILD_RETRIEVAL or not DEV_RETRIEVAL_MAXK_FILE.exists()
    reranker = CrossEncoderReranker(model_name=ACTIVE_CE_MODEL) if need_dev_reranker else None

retrieved_dev_maxk = run_retrieval(
    claims_path=DEV_FILE,
    out_path=DEV_RETRIEVAL_MAXK_FILE,
    evidence=EVIDENCE_TEXT,
    evidence_ids=EVIDENCE_IDS,
    word_vectorizer=WORD_VECTORIZER,
    word_matrix=WORD_MATRIX,
    char_vectorizer=CHAR_VECTORIZER,
    char_matrix=CHAR_MATRIX,
    reranker=reranker,
    k=MAX_K_RETRIEVE,
    force=FORCE_REBUILD_RETRIEVAL,
    stage1_scored_out_path=DEV_STAGE1_SCORED_FILE,
)

retrieved_dev_by_k = {}
for k in K_RETRIEVE_CHOICES:
    topk = {cid: list(evs[:k]) for cid, evs in retrieved_dev_maxk.items()}
    retrieved_dev_by_k[k] = topk
    save_json(topk, RETRIEVED_DEV_FILES_BY_K[k])
    print(f'[RET] cached dev top-{k} -> {RETRIEVED_DEV_FILES_BY_K[k]}')

retrieved_dev = retrieved_dev_by_k[K_RETRIEVE]
print('[RET] training uses gold evidence; development inference uses retrieved evidence.')

if reranker is not None:
    del reranker
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## Stage 3: Claim Classification

Prepare claim-evidence text pairs for the DeBERTa classifier.


In [ ]:
from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer,
    DataCollatorWithPadding, Trainer, TrainingArguments, set_seed,
)
from sklearn.metrics import accuracy_score, f1_score

SEP = ' [SEP] '


def join_evidence(evidence_ids, evidence: Dict[str, str], max_pieces: Optional[int] = None) -> str:
    max_pieces = K_RETRIEVE if max_pieces is None else max_pieces
    pieces = []
    for eid in list(evidence_ids or [])[:max_pieces]:
        if eid not in evidence:
            raise ValueError(f'Unknown evidence passage: {eid}')
        text = evidence[eid]
        if text:
            pieces.append(clean_text(text))
    return SEP.join(pieces)


def build_examples(claims: Dict[str, dict],
                   evidence: Dict[str, str],
                   *, use_evidences: Optional[Dict[str, list]] = None,
                   max_pieces: Optional[int] = None,
                   has_labels: bool = True) -> list:
    out = []
    for cid, c in claims.items():
        if use_evidences is None:
            ev_ids = c.get('evidences', [])
        else:
            if cid not in use_evidences or not use_evidences[cid]:
                raise ValueError(f'{cid}: retrieved evidence is missing or empty.')
            ev_ids = use_evidences[cid]
        item = {
            'claim_id': cid,
            'claim_text': clean_text(c.get('claim_text', '')),
            'evidence_text': join_evidence(ev_ids, evidence, max_pieces=max_pieces),
        }
        if has_labels:
            item['label'] = LABEL2ID[c['claim_label']]
        out.append(item)
    return out


def tokenize_batch(batch, tokenizer, max_length: int = MAX_LEN):
    enc = tokenizer(
        batch['claim_text'],
        batch['evidence_text'],
        padding=False,
        truncation=True,
        max_length=max_length,
    )
    if 'label' in batch:
        enc['labels'] = batch['label']
    return enc


print('[CLS] dataset builders ready')


## Train Classifier Utilities

Define class weighting, metrics, and the weighted Trainer used for claim classification.


In [ ]:
def class_weights(labels, n_classes):
    counts = np.bincount(labels, minlength=n_classes)
    counts = np.where(counts == 0, 1, counts)
    w = counts.sum() / (n_classes * counts)
    w = w / w.mean()
    return torch.tensor(w, dtype=torch.float32)


def compute_metrics(eval_pred):
    if hasattr(eval_pred, 'predictions'):
        logits = eval_pred.predictions
        labels = eval_pred.label_ids
    else:
        logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'macro_f1': f1_score(labels, preds, average='macro'),
    }


class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weight=None, **kwargs):
        super().__init__(*args, **kwargs)
        self._class_weight = class_weight
        # Newer Transformers may pass num_items_in_batch; our weighted CE uses mean reduction.
        self.model_accepts_loss_kwargs = False

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        weight = self._class_weight.to(logits.device) if self._class_weight is not None else None
        loss = torch.nn.functional.cross_entropy(logits, labels, weight=weight)
        return (loss, outputs) if return_outputs else loss

print('[CLS] weighted trainer utilities ready')


In [ ]:
set_all_seeds(); set_seed(SEED)

print('[CLS] loading data ...')
print('[CLS] architecture: classifier trains on all labelled gold evidence only; dev is all gold evidence only.')
train_claims = load_claims(TRAIN_FILE)
dev_claims   = load_claims(DEV_FILE)

train_examples = build_examples(
    train_claims,
    EVIDENCE_TEXT,
    use_evidences=None,
    max_pieces=GOLD_EVIDENCE_MAX_PIECES,
    has_labels=True,
)

dev_examples = build_examples(
    dev_claims,
    EVIDENCE_TEXT,
    use_evidences=None,
    max_pieces=GOLD_EVIDENCE_MAX_PIECES,
    has_labels=True,
)

raw_train_ds = Dataset.from_list(train_examples)
raw_dev_ds   = Dataset.from_list(dev_examples)
print(f'[CLS] train={len(train_examples)}  dev={len(dev_examples)}')
print('[CLS] raw columns:', raw_train_ds.column_names)


In [ ]:
print(f'[CLS] tokenizer + model: {CLS_MODEL}')

tokenizer = AutoTokenizer.from_pretrained(CLS_MODEL, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(
    CLS_MODEL,
    num_labels=len(LABELS),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
)

# Keep classifier fine-tuning in FP32; avoids GradScaler FP16 unscale errors.
model = model.float()

# Rebuild raw datasets every time this cell runs.
# This prevents KeyError: 'claim_text' if train_ds/dev_ds were already tokenized.
raw_train_ds = Dataset.from_list(train_examples)
raw_dev_ds   = Dataset.from_list(dev_examples)


def _tok(batch):
    return tokenize_batch(batch, tokenizer, max_length=MAX_LEN)


train_ds = raw_train_ds.map(
    _tok,
    batched=True,
    remove_columns=raw_train_ds.column_names,
)

dev_ds = raw_dev_ds.map(
    _tok,
    batched=True,
    remove_columns=raw_dev_ds.column_names,
)

collator = DataCollatorWithPadding(tokenizer=tokenizer)
cw = class_weights([ex['label'] for ex in train_examples], len(LABELS))
print(f'[CLS] class weights = {cw.tolist()}')
print('[CLS] tokenized datasets ready:', train_ds.column_names)

# Train classifier and save the best checkpoint.
import inspect

CLASSIFIER_DIR.mkdir(parents=True, exist_ok=True)


def make_training_args(kwargs: dict) -> TrainingArguments:
    """Build TrainingArguments while dropping args unsupported by the installed Transformers version."""
    kwargs = dict(kwargs)
    sig = inspect.signature(TrainingArguments.__init__)
    valid = set(sig.parameters)

    # Preserve an explicitly disabled evaluation strategy for the final train+dev fit.
    strategy = kwargs.pop('eval_strategy', kwargs.pop('evaluation_strategy', 'epoch'))
    # Transformers renamed this argument across versions.
    if 'eval_strategy' in valid:
        kwargs['eval_strategy'] = strategy
        kwargs.pop('evaluation_strategy', None)
    elif 'evaluation_strategy' in valid:
        kwargs['evaluation_strategy'] = strategy
        kwargs.pop('eval_strategy', None)
    else:
        kwargs.pop('eval_strategy', None)
        kwargs.pop('evaluation_strategy', None)

    filtered = {k: v for k, v in kwargs.items() if k in valid}
    dropped = sorted(set(kwargs) - set(filtered))
    if dropped:
        print('[ARGS] dropped unsupported TrainingArguments:', dropped)
    return TrainingArguments(**filtered)


def make_weighted_trainer(**kwargs) -> WeightedTrainer:
    """Build WeightedTrainer while adapting to Trainer API changes such as tokenizer -> processing_class."""
    class_weight = kwargs.pop('class_weight', None)
    sig = inspect.signature(Trainer.__init__)
    valid = set(sig.parameters)

    tok = kwargs.pop('tokenizer', None)
    if tok is not None:
        if 'tokenizer' in valid:
            kwargs['tokenizer'] = tok
        elif 'processing_class' in valid:
            kwargs['processing_class'] = tok
        else:
            print('[TRAINER] dropped unsupported tokenizer argument')

    filtered = {k: v for k, v in kwargs.items() if k in valid}
    dropped = sorted(set(kwargs) - set(filtered))
    if dropped:
        print('[TRAINER] dropped unsupported Trainer args:', dropped)
    return WeightedTrainer(**filtered, class_weight=class_weight)


def select_best_epoch_by_accuracy(trainer, default_epochs: int = EPOCHS) -> tuple[int, List[dict]]:
    """Pick the epoch with highest dev accuracy; ties use macro-F1, then lower eval loss."""
    rows = []
    for item in trainer.state.log_history:
        if 'eval_accuracy' not in item:
            continue
        epoch_float = float(item.get('epoch', len(rows) + 1))
        row = {
            'epoch': int(round(epoch_float)),
            'epoch_float': epoch_float,
            'eval_accuracy': float(item.get('eval_accuracy', 0.0)),
            'eval_macro_f1': float(item.get('eval_macro_f1', -1.0)),
            'eval_loss': float(item.get('eval_loss', float('inf'))),
        }
        rows.append(row)

    if not rows:
        print(f'[CLS] no epoch-level eval logs found; using EPOCHS={default_epochs}')
        return int(default_epochs), rows

    best = max(rows, key=lambda r: (r['eval_accuracy'], r['eval_macro_f1'], -r['eval_loss'], -r['epoch']))
    best_epochs = max(1, int(best['epoch']))
    print('[CLS] epoch selection by dev accuracy:')
    print('epoch\taccuracy\tmacro_f1\teval_loss')
    for row in rows:
        mark = '  <== selected' if row is best else ''
        print(f"{row['epoch']}\t{row['eval_accuracy']:.6f}\t{row['eval_macro_f1']:.6f}\t{row['eval_loss']:.6f}{mark}")
    if best_epochs >= int(default_epochs):
        print(f'[CLS] best epoch is the last tested epoch ({best_epochs}); consider trying EPOCHS=20 later if time allows.')
    return best_epochs, rows


steps_per_epoch = math.ceil(len(train_ds) / max(1, BATCH_SIZE * GRAD_ACCUM_STEPS))
warmup_steps = max(1, int(WARMUP_RATIO * steps_per_epoch * EPOCHS))
print(f'[ARGS] warmup_steps={warmup_steps}')

training_kwargs = dict(
    output_dir=str(CLASSIFIER_DIR),
    overwrite_output_dir=True,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=warmup_steps,
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    greater_is_better=True,
    logging_steps=50,
    fp16=False,
    bf16=False,
    report_to='none',
    save_total_limit=2,
    seed=SEED,
)

# Reassert FP32 before Trainer wraps the model.
model = model.float()

trainer = make_weighted_trainer(
    model=model,
    args=make_training_args(training_kwargs),
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
    class_weight=cw,
)

print('[CLS] starting training ...')
trainer.train()
BEST_CLASSIFIER_EPOCHS, classifier_epoch_rows = select_best_epoch_by_accuracy(trainer, default_epochs=EPOCHS)
save_json(classifier_epoch_rows, CLASSIFIER_DIR / 'epoch_selection_by_accuracy.json', indent=2)
save_json({'best_epochs': BEST_CLASSIFIER_EPOCHS}, CLASSIFIER_DIR / 'best_epoch.json', indent=2)
print(f'[CLS] BEST_CLASSIFIER_EPOCHS={BEST_CLASSIFIER_EPOCHS}; final train+dev will use this many epochs.')

print('[CLS] saving best model ...')
trainer.save_model(str(CLASSIFIER_DIR))
tokenizer.save_pretrained(str(CLASSIFIER_DIR))

dev_metrics = trainer.evaluate()
dev_metrics['best_classifier_epochs'] = BEST_CLASSIFIER_EPOCHS
print('[CLS] final dev metrics:', json.dumps(dev_metrics, indent=2))
save_json(dev_metrics, CLASSIFIER_DIR / 'metrics.json')


## Predict Claim Labels

Load the trained classifier and predict labels from retrieved evidence.


In [ ]:
@torch.inference_mode()
def predict_labels(claims_path: Path,
                   retrieved: Dict[str, list],
                   model_dir: Path = CLASSIFIER_DIR,
                   batch_size: int = EVAL_BATCH_SIZE,
                   max_len: int = MAX_LEN) -> Dict[str, str]:
    model_dir = Path(model_dir)
    if not model_dir.exists():
        raise FileNotFoundError(f'Model directory does not exist yet: {model_dir}. Run the classifier training cell first.')

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'[PRED] device={device}  model_dir={model_dir}')
    tok = AutoTokenizer.from_pretrained(model_dir)
    mdl = AutoModelForSequenceClassification.from_pretrained(model_dir).to(device).eval()

    claims_loaded = load_claims(claims_path)
    examples = build_examples(claims_loaded, EVIDENCE_TEXT,
                              use_evidences=retrieved, has_labels=False)
    ids = [ex['claim_id'] for ex in examples]

    ds = Dataset.from_list(examples)

    def _tok(batch):
        return tokenize_batch(batch, tok, max_length=max_len)

    ds = ds.map(_tok, batched=True, remove_columns=ds.column_names)

    coll = DataCollatorWithPadding(tokenizer=tok)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                        num_workers=0, collate_fn=coll)
    preds = []
    for batch in tqdm(loader, desc='classify'):
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        logits = mdl(**batch).logits
        preds.extend(logits.argmax(-1).cpu().numpy().tolist())
    return {cid: ID2LABEL[int(p)] for cid, p in zip(ids, preds)}


print('[PRED] prediction helper ready')


# 3. Testing and Evaluation


## Select the Number of Evidence Sentences

Compare different K values on the dev set and choose the best retrieval F1.


In [ ]:
def retrieval_metrics(gold_claims: Dict[str, dict],
                      retrieved: Dict[str, list]) -> dict:
    p_list, r_list, f_list = [], [], []
    for cid, c in gold_claims.items():
        gold = set(c.get('evidences', []))
        pred = set(retrieved.get(cid, []))
        if not gold or not pred:
            p_list.append(0.0); r_list.append(0.0); f_list.append(0.0); continue
        tp = len(gold & pred)
        prec = tp / len(pred) if pred else 0
        rec  = tp / len(gold) if gold else 0
        f    = (2 * prec * rec / (prec + rec)) if (prec + rec) else 0.0
        p_list.append(prec); r_list.append(rec); f_list.append(f)
    return {'precision': float(np.mean(p_list)), 'recall': float(np.mean(r_list)), 'f1': float(np.mean(f_list))}


def stage1_recall_metrics(gold_claims: Dict[str, dict],
                          scored_candidates: Dict[str, List[dict]],
                          cutoffs: Sequence[int] = CANDIDATE_POOL_SIZE_GRID) -> Dict[str, dict]:
    metrics = {}
    for cutoff in cutoffs:
        recalls, hits, all_gold, counts = [], [], [], []
        for cid, claim in gold_claims.items():
            gold = set(claim.get('evidences', []))
            pred = [item['evidence_id'] for item in scored_candidates.get(cid, [])[:cutoff]]
            pred_set = set(pred)
            counts.append(len(pred))
            if not gold:
                recalls.append(0.0); hits.append(0.0); all_gold.append(0.0); continue
            overlap = len(gold & pred_set)
            recalls.append(overlap / len(gold))
            hits.append(1.0 if overlap > 0 else 0.0)
            all_gold.append(1.0 if gold.issubset(pred_set) else 0.0)
        metrics[f'@{cutoff}'] = {
            'mean_gold_recall': float(np.mean(recalls)),
            'hit_rate_at_least_one_gold': float(np.mean(hits)),
            'all_gold_recovered_rate': float(np.mean(all_gold)),
            'avg_candidate_count': float(np.mean(counts)),
        }
    return metrics


def print_stage1_metrics_table(metrics: Dict[str, dict]) -> None:
    print('cutoff\tmean_recall\thit_rate\tall_gold\tavg_candidates')
    for cutoff, values in metrics.items():
        print(
            f"{cutoff}\t"
            f"{values['mean_gold_recall']:.4f}\t"
            f"{values['hit_rate_at_least_one_gold']:.4f}\t"
            f"{values['all_gold_recovered_rate']:.4f}\t"
            f"{values['avg_candidate_count']:.1f}"
        )


dev_claims_full = load_claims(DEV_FILE)

if 'retrieved_dev_by_k' not in globals() or not retrieved_dev_by_k:
    retrieved_dev_by_k = {}
    for k in K_RETRIEVE_CHOICES:
        path = RETRIEVED_DEV_FILES_BY_K[k]
        if path.exists():
            retrieved_dev_by_k[k] = load_json(path)

print('Dev retrieval sweep; selecting K by retrieval F1:')
print('k\tprecision\trecall\tf1')
retrieval_sweep_rows = []
for k in K_RETRIEVE_CHOICES:
    if k not in retrieved_dev_by_k:
        raise FileNotFoundError(f'Missing dev retrieval for k={k}: {RETRIEVED_DEV_FILES_BY_K[k]}')
    metrics = retrieval_metrics(dev_claims_full, retrieved_dev_by_k[k])
    row = {'k': int(k), **metrics}
    retrieval_sweep_rows.append(row)
    print(f"{k}\t{metrics['precision']:.6f}\t{metrics['recall']:.6f}\t{metrics['f1']:.6f}")

best_row = max(retrieval_sweep_rows, key=lambda r: (r['f1'], r['recall'], -abs(r['k'] - 6)))
BEST_K_RETRIEVE = int(best_row['k'])
K_RETRIEVE = BEST_K_RETRIEVE
retrieved_dev = retrieved_dev_by_k[K_RETRIEVE]
RETRIEVED_DEV_FILE = RETRIEVED_DEV_FILES_BY_K[K_RETRIEVE]
RETRIEVED_TEST_FILE = RETRIEVED_TEST_FILES_BY_K[K_RETRIEVE]

save_json(retrieval_sweep_rows, ARTIFACT_DIR / 'dev_retrieval_k_sweep.json', indent=2)
print()
print(f'[RET] selected K_RETRIEVE={K_RETRIEVE} by dev retrieval F1={best_row["f1"]:.6f}')
print(f'[RET] classifier/final pipeline will use -> {RETRIEVED_DEV_FILE}')

if DEV_STAGE1_SCORED_FILE.exists():
    dev_stage1_scored = load_json(DEV_STAGE1_SCORED_FILE)
    dev_stage1_metrics = stage1_recall_metrics(dev_claims_full, dev_stage1_scored)
    save_json(dev_stage1_metrics, DEV_STAGE1_METRICS_FILE)
    print()
    print('Stage-1 candidate recall diagnostics:')
    print_stage1_metrics_table(dev_stage1_metrics)


## Generate Dev Predictions

Evaluate the train-only classifier on retrieved development evidence before the optional final fit.


In [ ]:
def run_pipeline(claims_path: Path, out_path: Path,
                 retrieved: Dict[str, list],
                 model_dir: Path = CLASSIFIER_DIR) -> Dict[str, dict]:
    labels = predict_labels(claims_path, retrieved, model_dir=model_dir)
    claims_loaded = load_claims(claims_path)
    submission = {}
    for cid in claims_loaded:
        evs = retrieved.get(cid, [])
        if not evs:
            raise ValueError(f'{cid}: retrieved evidence is missing or empty.')
        submission[cid] = {
            'claim_label': labels.get(cid, 'NOT_ENOUGH_INFO'),
            'evidences': list(evs),
        }
    validate_submission(claims_loaded, submission)
    save_json(submission, out_path)
    print(f'[PIPE] wrote submission -> {out_path}')
    return submission


def train_final_classifier_on_train_plus_dev() -> None:
    print('[FINAL] training classifier on train + dev ...')
    set_all_seeds(); set_seed(SEED)
    FINAL_CLASSIFIER_DIR.mkdir(parents=True, exist_ok=True)

    final_train_claims = {}
    final_train_claims.update(load_claims(TRAIN_FILE))
    final_train_claims.update(load_claims(DEV_FILE))

    final_train_examples = build_examples(
        final_train_claims,
        EVIDENCE_TEXT,
        use_evidences=None,
        max_pieces=GOLD_EVIDENCE_MAX_PIECES,
        has_labels=True,
    )
    print(f'[FINAL] training examples={len(final_train_examples)}; no overlapping dev evaluation')
    tok = AutoTokenizer.from_pretrained(CLS_MODEL, use_fast=True)
    mdl = AutoModelForSequenceClassification.from_pretrained(
        CLS_MODEL,
        num_labels=len(LABELS),
        id2label=ID2LABEL,
        label2id=LABEL2ID,
        ignore_mismatched_sizes=True,
    )
    mdl = mdl.float()

    def _tok(batch):
        return tokenize_batch(batch, tok, max_length=MAX_LEN)

    final_raw_train_ds = Dataset.from_list(final_train_examples)
    final_train_ds = final_raw_train_ds.map(_tok, batched=True, remove_columns=final_raw_train_ds.column_names)

    final_epochs = int(globals().get('BEST_CLASSIFIER_EPOCHS', EPOCHS))
    final_epochs = max(1, final_epochs)
    final_steps_per_epoch = math.ceil(len(final_train_ds) / max(1, BATCH_SIZE * GRAD_ACCUM_STEPS))
    final_warmup_steps = max(1, int(WARMUP_RATIO * final_steps_per_epoch * final_epochs))
    print(f'[FINAL] epochs={final_epochs} selected from train/dev accuracy sweep')
    print(f'[FINAL] warmup_steps={final_warmup_steps}')

    kwargs = dict(
        output_dir=str(FINAL_CLASSIFIER_DIR),
        overwrite_output_dir=True,
        num_train_epochs=final_epochs,
        learning_rate=LR,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        weight_decay=WEIGHT_DECAY,
        warmup_steps=final_warmup_steps,
        save_strategy='epoch',
        eval_strategy='no',
        load_best_model_at_end=False,
        logging_steps=50,
        fp16=False,
        bf16=False,
        report_to='none',
        save_total_limit=2,
        seed=SEED,
    )

    final_trainer = make_weighted_trainer(
        model=mdl,
        args=make_training_args(kwargs),
        train_dataset=final_train_ds,
        tokenizer=tok,
        data_collator=DataCollatorWithPadding(tokenizer=tok),
        class_weight=class_weights([ex['label'] for ex in final_train_examples], len(LABELS)),
    )
    final_trainer.train()
    final_trainer.save_model(str(FINAL_CLASSIFIER_DIR))
    tok.save_pretrained(str(FINAL_CLASSIFIER_DIR))
    save_json(
        {'training_split': 'train+dev', 'epochs': final_epochs, 'held_out_evaluation': False},
        FINAL_CLASSIFIER_DIR / 'training_summary.json',
    )
    del final_trainer, mdl
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


print('[PIPE] Development prediction with the train-only classifier.')
dev_submission = run_pipeline(DEV_FILE, DEV_PREDICTIONS_FILE, retrieved_dev, model_dir=CLASSIFIER_DIR)
print_elapsed_hours('[TIME] total elapsed after development pipeline')


## Evaluate Dev Results

Compute evidence F-score, claim accuracy, and their harmonic mean.


In [ ]:
# Evaluate dev predictions directly in the notebook and compare against the official baseline.
# Metric definitions follow the official evaluator. Require complete predictions before scoring.

def evaluate_predictions(predictions_path: Path, groundtruth_path: Path) -> dict:
    predictions_path = Path(predictions_path)
    groundtruth_path = Path(groundtruth_path)

    if not predictions_path.exists():
        raise FileNotFoundError(
            f'Missing prediction file: {predictions_path}. Run the pipeline cell before evaluation.'
        )
    if not groundtruth_path.exists():
        raise FileNotFoundError(f'Missing groundtruth file: {groundtruth_path}')

    predictions = load_json(predictions_path)
    groundtruth = load_json(groundtruth_path)
    if not groundtruth:
        raise ValueError('Ground-truth claims cannot be empty.')
    validate_submission(groundtruth, predictions)

    f_scores, accuracies = [], []
    evaluated = 0

    for claim_id, claim in sorted(groundtruth.items()):
        pred = predictions[claim_id]

        evaluated += 1
        accuracies.append(1.0 if pred['claim_label'] == claim['claim_label'] else 0.0)

        evidence_fscore = 0.0
        if isinstance(pred['evidences'], list) and len(pred['evidences']) > 0:
            pred_evs = set(pred['evidences'])
            gold_evs = claim.get('evidences', [])
            evidence_correct = sum(1 for ev in gold_evs if ev in pred_evs)
            if evidence_correct > 0:
                evidence_recall = evidence_correct / max(1, len(gold_evs))
                evidence_precision = evidence_correct / max(1, len(pred['evidences']))
                evidence_fscore = (2 * evidence_precision * evidence_recall) / (evidence_precision + evidence_recall)
        f_scores.append(evidence_fscore)

    mean_f = float(np.mean(f_scores if f_scores else [0.0]))
    mean_acc = float(np.mean(accuracies if accuracies else [0.0]))
    hmean = 0.0 if (mean_f == 0.0 and mean_acc == 0.0) else (2 * mean_f * mean_acc) / (mean_f + mean_acc)

    return {
        'evaluated_claims': evaluated,
        'total_claims': len(groundtruth),
        'evidence_fscore': mean_f,
        'claim_accuracy': mean_acc,
        'harmonic_mean': hmean,
    }


def print_eval_block(name: str, metrics: dict) -> None:
    print(f'[{name}] evaluated: {metrics["evaluated_claims"]} / {metrics["total_claims"]} claims')
    print(f'[{name}] Evidence Retrieval F-score (F)    = {metrics["evidence_fscore"]:.6f}')
    print(f'[{name}] Claim Classification Accuracy (A) = {metrics["claim_accuracy"]:.6f}')
    print(f'[{name}] Harmonic Mean of F and A          = {metrics["harmonic_mean"]:.6f}')


baseline_file = DATA_DIR / 'dev-claims-baseline.json'
my_metrics = evaluate_predictions(DEV_PREDICTIONS_FILE, DEV_FILE)
base_metrics = evaluate_predictions(baseline_file, DEV_FILE)

print(f'[EVAL] my predictions : {DEV_PREDICTIONS_FILE}')
print(f'[EVAL] baseline file  : {baseline_file}')
print(f'[EVAL] groundtruth    : {DEV_FILE}')
print()
print_eval_block('MINE', my_metrics)
print()
print_eval_block('BASELINE', base_metrics)
print()

margin = my_metrics['harmonic_mean'] - base_metrics['harmonic_mean']
if margin > 0:
    print(f'[RESULT] Beats baseline by H = +{margin:.6f}')
elif margin < 0:
    print(f'[RESULT] Below baseline by H = {margin:.6f}')
else:
    print('[RESULT] Ties baseline on H')


## Train Final Model and Predict Test Set

Train the final classifier on train+dev labels and generate test predictions.


In [ ]:
if RUN_TEST_SUBMISSION:
    print('[TEST] RUN_TEST_SUBMISSION=True, retrieving and predicting test split.')
    test_ce_model = globals().get('ACTIVE_CE_MODEL', CE_MODEL)
    if CE_RERANK_WEIGHT <= 0:
        print('[TEST] Stage-2 disabled; test retrieval uses Stage-1 TF-IDF ranking only.')
        test_reranker = None
    else:
        test_reranker = CrossEncoderReranker(model_name=test_ce_model)
    retrieved_test = run_retrieval(
        claims_path=TEST_FILE,
        out_path=RETRIEVED_TEST_FILE,
        evidence=EVIDENCE_TEXT,
        evidence_ids=EVIDENCE_IDS,
        word_vectorizer=WORD_VECTORIZER,
        word_matrix=WORD_MATRIX,
        char_vectorizer=CHAR_VECTORIZER,
        char_matrix=CHAR_MATRIX,
        reranker=test_reranker,
        k=K_RETRIEVE,
        force=FORCE_REBUILD_RETRIEVAL,
        stage1_scored_out_path=TEST_STAGE1_SCORED_FILE,
    )
    if test_reranker is not None:
        del test_reranker
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    train_final_classifier_on_train_plus_dev()
    test_submission = run_pipeline(TEST_FILE, TEST_PREDICTIONS_FILE, retrieved_test, model_dir=FINAL_CLASSIFIER_DIR)
else:
    print('[TEST] skipped. Set RUN_TEST_SUBMISSION=True in the config cell for final submission generation.')

print_elapsed_hours('[TIME] total elapsed after final train+dev/test pipeline')


## Create Leaderboard Submission

Package `test-output.json` into the `submission.zip` file for submission.


In [ ]:
import zipfile
from pathlib import Path

try:
    from google.colab import files
    _HAS_COLAB_FILES = True
except Exception:
    files = None
    _HAS_COLAB_FILES = False

zip_path = SUBMISSION_ZIP_FILE

if not TEST_PREDICTIONS_FILE.exists():
    print(f'[ZIP] {TEST_PREDICTIONS_FILE} does not exist yet. Set RUN_TEST_SUBMISSION=True and rerun the final pipeline cell first.')
else:
    if zip_path.exists():
        raise FileExistsError(f'{zip_path} already exists. Choose a new output path to preserve it.')

    with zipfile.ZipFile(zip_path, 'x', zipfile.ZIP_DEFLATED) as zf:
        zf.write(TEST_PREDICTIONS_FILE, arcname='test-output.json')

    print(f'[ZIP] Created: {zip_path} ({zip_path.stat().st_size:,} bytes)')
    if _HAS_COLAB_FILES:
        print('Starting download to your browser...')
        files.download(str(zip_path))
    else:
        print('[ZIP] Local run detected; submit this zip file manually.')

if 'print_elapsed_hours' in globals():
    print_elapsed_hours('[TIME] total elapsed after zip cell')
